# Grammar emptiness

Compiles a JSON Schema fragment, runs the tokenizer x grammar emptiness check, and replays the witness through the compiled DFA.


The code cell below builds a minimal in-memory artifact, runs the real PromptABI checker, and asserts the proof obligation.


In [ ]:
import json
import tempfile
from pathlib import Path

from promptabi import ArtifactKind, ArtifactLocation, SchemaArtifact, TokenizerArtifact
from promptabi import analyze_tokenizer_grammar_emptiness, prove_grammar_emptiness
from promptabi.json_schema import compile_json_schema_mapping
from promptabi.proof_sketches import ProofOutcome
from promptabi.tokenizers import ByteLevelTokenizer

schema = {"type": "object", "additionalProperties": False}
compiled = compile_json_schema_mapping(schema)
with tempfile.TemporaryDirectory() as tempdir:
    schema_path = Path(tempdir) / "empty-object.schema.json"
    schema_path.write_text(json.dumps(schema), encoding="utf-8")
    tokenizer_artifact = TokenizerArtifact(
        kind=ArtifactKind.TOKENIZER,
        name="byte",
        location=ArtifactLocation(uri="memory://byte-tokenizer"),
        family="byte-level",
    )
    schema_artifact = SchemaArtifact(
        kind=ArtifactKind.SCHEMA,
        name="empty-object",
        location=ArtifactLocation(path=str(schema_path)),
    )
    report = analyze_tokenizer_grammar_emptiness(tokenizer_artifact, schema_artifact, ByteLevelTokenizer())
    sketch = prove_grammar_emptiness(report, automaton=compiled.grammar.automaton)

assert sketch.outcome is ProofOutcome.PROVEN
assert sketch.passed
sketch.to_dict()
